# Notebook 03 — Inspect Rubrics + Gold Standard

## Why this notebook exists

Notebook 02 covered the **inputs** to the eval: the task set. This notebook covers the two artifacts that turn agent output into a graded number:

1. **Rubrics** (`scaffolding/ground_truth/<task_id>.md`) — one per non-nav-only task. Each rubric breaks the review into 3-5 binary pass/fail dimensions (correctness, scope_discipline, etc.) plus a list of red flags. The LLM judge in `pr_reviewer/` reads the rubric and grades a diff dimension-by-dimension.
2. **Gold standard** (`scaffolding/gold_standard/*.yaml` + `diffs/*.diff`) — small set of hand-authored "good diff" and "bad diff" examples paired with a *human-authored verdict*. This is the calibration anchor: notebook 04 measures whether the LLM judge agrees with the human verdicts. Below 80% agreement, you don't trust the judge.

These are the most labor-intensive artifacts in the workshop and the ones most vulnerable to author bias if you let an LLM write them. We ship them prebuilt; you inspect and learn the schema.

This notebook walks you through:

- **Step 1**: Validate the prebuilt rubrics structurally and confirm one rubric per non-nav-only task.
- **Step 2**: Read one rubric end-to-end and understand the dimension/red-flag pattern.
- **Step 3**: Validate the prebuilt gold-standard set.
- **Step 4**: Read a "good diff" + "bad diff" pair side-by-side and see how the human verdicts map to rubric dimensions.
- **Step 5**: Reference the rubric and gold-entry schemas, so you can author them for your own repo.

Like notebook 02, you will **not** be writing or modifying anything here. The deliverable is understanding.

## Step 1 — Validate the prebuilt rubrics

The validator does two things:

1. **Schema check**: every `*.md` in `scaffolding/ground_truth/` parses as a rubric — has a title, a `## Dimensions` section with 3+ named dimensions, and a `## Red flags` section.
2. **Coverage check**: there's exactly one rubric file per non-nav-only task. Nav-only tasks (`T08`, `T09`) have no diff to review, so they're skipped.

If the prebuilt set is intact, both should pass without errors.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '.')
from validators.rubrics import validate_rubric, validate_rubric_matches_tasks

WORKSHOP_DIR = Path.cwd().resolve()
TASKS_FILE = WORKSHOP_DIR / 'scaffolding' / 'tasks' / 'tasks.yaml'
GROUND_TRUTH_DIR = WORKSHOP_DIR / 'scaffolding' / 'ground_truth'
GOLD_DIR = WORKSHOP_DIR / 'scaffolding' / 'gold_standard'
DIFFS_DIR = GOLD_DIR / 'diffs'
RUBRIC_SCHEMA = WORKSHOP_DIR / 'scaffolding' / 'rubric_schema_example.md'
GOLD_SCHEMA = WORKSHOP_DIR / 'scaffolding' / 'gold_entry_schema_example.yaml'

cross = validate_rubric_matches_tasks(GROUND_TRUTH_DIR, TASKS_FILE)
print(cross.report())
print()

all_good = cross.passed
for path in sorted(GROUND_TRUTH_DIR.glob('*.md')):
    v = validate_rubric(path)
    print(v.report())
    print()
    all_good = all_good and v.passed

assert all_good, 'Rubric validation failed. See errors above.'

## Step 2 — Read one rubric end-to-end

Below is `T03_hardcoded_region_in_kb_client.md`, the rubric for the medium task you saw in notebook 02. Pay attention to:

- **Dimensions are binary and concrete.** Each one is a list of bullets that yield a clean pass/fail. No "mostly correct" or "good effort" qualifiers.
- **Scope discipline is its own dimension.** Drive-by edits ("while I'm here, I added type hints") are a separate axis from correctness — a PR can be technically correct on the main fix but fail scope discipline.
- **Red flags are real failure modes you've seen before**, not generic checklist items. They give the LLM judge concrete patterns to look for.

In [ ]:
from IPython.display import Markdown, display

featured = GROUND_TRUTH_DIR / 'T03_hardcoded_region_in_kb_client.md'
display(Markdown(featured.read_text()))

### Note: trap-task rubrics look different

A normal-task rubric (like T03 above) describes what a successful diff looks like. A **trap-task** rubric instead describes what a successful *non-diff* looks like — the issue describes a fictional bug, so any non-empty diff is a fail. Below: T06's rubric.

In [ ]:
trap_rubric = GROUND_TRUTH_DIR / 'T06_trap_silent_exception_in_cache.md'
display(Markdown(trap_rubric.read_text()))

## Step 3 — Validate the prebuilt gold-standard set

The gold standard is what makes the LLM judge trustworthy. Each entry pairs:

- a **synthetic diff** (`scaffolding/gold_standard/diffs/*.diff`) — hand-authored to deliberately exercise specific dimensions of a rubric.
- a **per-dimension human verdict** (`scaffolding/gold_standard/*.yaml`) — what *we* (the curator) say each dimension should be graded.

In notebook 04, the LLM judge will grade the same diffs against the same rubrics, and we'll measure agreement. Below 80% agreement, you don't trust the judge.

The prebuilt set covers 4 tasks (T01, T02, T03, T05) with a **good diff + bad diff pair** for each — 8 entries total. The good entries pass every dimension; the bad entries deliberately violate red flags so we can see whether the judge catches them.

In [ ]:
from validators.gold_standard import validate_gold_entry

gold_files = sorted(GOLD_DIR.glob('*.yaml'))
print(f'Found {len(gold_files)} gold entries.\n')

all_good = True
for p in gold_files:
    v = validate_gold_entry(p)
    print(v.report())
    print()
    all_good = all_good and v.passed

assert all_good, 'Gold-standard validation failed.'

## Step 4 — Read a good/bad pair side-by-side

Let's look at the T03 pair: same task, two different diffs, dramatically different verdicts. This is the shape of a useful gold entry — it makes the rubric distinguish between *acceptable* and *unacceptable* outcomes for the same task.

In [ ]:
import yaml

def show_entry(yaml_path: Path, label: str) -> None:
    entry = yaml.safe_load(yaml_path.read_text())
    diff_path = (yaml_path.parent / entry['diff_path']).resolve()
    diff_text = diff_path.read_text()

    verdicts_md = '\n'.join(
        f'- **{dim}**: `{verdict}`' for dim, verdict in entry['human_verdicts'].items()
    )
    red_flags_md = '\n'.join(
        f'- {rf}' for rf in entry.get('human_red_flags_hit') or []
    ) or '_(none)_'

    display(Markdown(f'''### {label} — `{entry['pr_slug']}`

**{entry['pr_title']}**

**Human verdicts:**

{verdicts_md}

**Red flags hit:**

{red_flags_md}

**Notes:**

{entry.get('notes', '_(none)_').strip()}
'''))
    print('--- Diff ---')
    print(diff_text)
    print()

show_entry(GOLD_DIR / 'T03_aws_region_clean.yaml', 'GOOD diff (passes all dimensions)')
show_entry(GOLD_DIR / 'T03_new_envvar_and_drive_by.yaml', 'BAD diff (fails all dimensions)')

## Step 5 — Schema reference

When you adapt this workshop to your own repo, you'll need to write rubrics and gold entries for your own tasks. The relevant schemas:

### Rubric (`scaffolding/ground_truth/<task_id>.md`)

```markdown
# <task_id> — <one-line title>

**Scope**: `<single file path>`

(brief context — what the task is, why this rubric exists)

## Dimensions

### 1. correctness
- <bullet describing a specific, binary criterion>
- <another bullet>

### 2. scope_discipline
- Only `<file>` is modified.
- No edits elsewhere.

### 3. <task-specific quality dimension>
- ...

## Red flags (any one → overall fail)

- <Specific failure mode #1>
- <Specific failure mode #2>
```

Rules:
- 3-5 dimensions. Always include `correctness` and `scope_discipline`. The third+ are task-specific (`logging_setup`, `resilience_quality`, `error_handling_quality`, etc.).
- Every dimension is a list of **binary** criteria. Avoid "should be reasonable" / "preferably". Use specific paths, line numbers, function names.
- Red flags are **patterns**, not generic warnings. "Adds a new dependency" beats "doesn't follow best practices".

### Gold entry (`scaffolding/gold_standard/<slug>.yaml` + `diffs/<slug>.diff`)

```yaml
pr_slug: <short_lowercase_slug>
pr_number: <int>           # Real PR number, or a synthetic ID like 1001+
pr_title: "..."

rubric_path: ../ground_truth/<task_id>.md
diff_path: ./diffs/<slug>.diff

human_verdicts:
  correctness: pass         # or fail — one entry per rubric dimension
  scope_discipline: pass
  <other_dim>: pass

human_red_flags_hit: []     # list of red-flag strings the human reviewer thinks are hit

notes: |
  Why this entry exists; what's interesting about the borderline.
```

Rules:
- **The dimension keys must match the rubric's dimensions exactly** — the validator cross-checks this.
- Aim for 1 GOOD + 1 BAD diff per task you cover. The bad one should violate a *specific* red flag, not just be sloppy.
- `human_verdicts` is YOUR judgment, not the agent's. The whole point is to have an oracle.

### Curating tips

- **Synthetic diffs are fine.** You don't need to find real merged PRs. The diff just has to look like a real PR — context lines around hunks, plausible file paths. The judge doesn't know whether the diff was hand-crafted.
- **Pair good/bad diffs against the SAME rubric.** That's how you find out whether the rubric actually distinguishes them.
- **Borderline entries are gold.** A diff where one dimension legitimately could go either way exposes whether the rubric's criteria are sharp enough.

## Next

Move on to **`04 calibrate the reviewer.ipynb`** to run the LLM judge against the gold standard and measure agreement.